In [1]:
### Importing libraries
import pandas as pd
import os
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import gzip
import shutil
import re

In [2]:
### Setting directory
current_directory = os.getcwd()
print("Current working directory:", current_directory)

new_directory = "C:/Users/tmfmo/Downloads/data_source"


os.chdir(new_directory)


current_directory = os.getcwd()
print("New working directory:", current_directory)

Current working directory: c:\Users\tmfmo\Downloads\data_cleaning
New working directory: C:\Users\tmfmo\Downloads\data_source


In [3]:
### Opening OpenAddress file

### Decompressing the file
with gzip.open("source.geojson.gz", "rb") as f_in:
    with open("source.geojson", "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)


gdf = gpd.read_file("source.geojson")

print(gdf.head())
print(gdf.isnull().sum())
print(gdf.columns)



               hash number              street unit    city  district region  \
0  2dc1dbcdfd558ecd     51           Parish Dr       BERLIN  Hartford     CT   
1  97e9c8baddd3b170     90  Stockings Brook Rd       BERLIN  Hartford     CT   
2  6960824f145050dc     99      Lamentation Dr       BERLIN  Hartford     CT   
3  eb15b7e40bc21073    207      Lamentation Dr       BERLIN  Hartford     CT   
4  8519d7523177658b    152     Spruce Brook Rd       BERLIN  Hartford     CT   

  postcode id                    geometry  
0    06037     POINT (-72.77387 41.63328)  
1    06037     POINT (-72.81025 41.59926)  
2    06037       POINT (-72.745 41.59919)  
3    06037      POINT (-72.7407 41.59928)  
4    06037     POINT (-72.74827 41.59935)  
hash        0
number      0
street      0
unit        0
city        0
district    0
region      0
postcode    0
id          0
geometry    0
dtype: int64
Index(['hash', 'number', 'street', 'unit', 'city', 'district', 'region',
       'postcode', 'id', 'geo

In [75]:
### Cleaning the Address Column
gdf = gdf.drop_duplicates(subset='street')
gdf = gdf.rename(columns={'geometry': 'gdf_geometry'})


gdf['cleaned_street'] = (
    gdf['street']
    .str.lower()
    .str.strip()
    .str.replace(r'\(\d+\)', '', regex=True)    # Removes any parenthesis with numbers
    .str.replace(r'\d+', '', regex=True)
    .str.replace(r'\b[a-zA-Z]\b', '', regex=True)   #Removes one letter words
    .str.replace('  ',' ')
    .drop_duplicates()
    .dropna()
    .str.strip()
)
gdf = gdf.drop_duplicates(subset='cleaned_street')

In [76]:
### Loading Census Data
zcta = gpd.read_file("C:/Users/tmfmo/Downloads/tl_2020_us_zcta520")

print(zcta.head())
print(zcta.columns)




  ZCTA5CE20 GEOID20 CLASSFP20 MTFCC20 FUNCSTAT20    ALAND20  AWATER20  \
0     35592   35592        B5   G6350          S  298552385    235989   
1     35616   35616        B5   G6350          S  559506992  41870756   
2     35621   35621        B5   G6350          S  117838488    409438   
3     35651   35651        B5   G6350          S  104521045    574316   
4     36010   36010        B5   G6350          S  335675180    236811   

    INTPTLAT20    INTPTLON20  \
0  +33.7427261  -088.0973903   
1  +34.7395036  -088.0193814   
2  +34.3350314  -086.7270557   
3  +34.4609087  -087.4801507   
4  +31.6598950  -085.8128958   

                                            geometry  
0  POLYGON ((-88.24735 33.6539, -88.24713 33.6541...  
1  POLYGON ((-88.13997 34.58184, -88.13995 34.582...  
2  POLYGON ((-86.81659 34.3496, -86.81648 34.3496...  
3  POLYGON ((-87.53087 34.42492, -87.53082 34.429...  
4  POLYGON ((-85.95712 31.67744, -85.95676 31.677...  
Index(['ZCTA5CE20', 'GEOID20', 'CLASSF

In [77]:
### Converting format of DF coordinates
def parse_location(point_str):
        if pd.isnull(point_str):
                return None
        coords = point_str.replace("POINT (", "").replace(")", "").split()
        lon, lat = map(float, coords)
        return Point(lon, lat)

In [78]:
### Converting to GeoDataFrame and merging Census data
def convert(df):
    gdf_points = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')
    
    zcta_merged = gpd.sjoin(
        gdf_points,
        zcta[['ZCTA5CE20', 'geometry']],
        how='left',
        predicate='within',
        lsuffix='left',
        rsuffix='zcta'  # avoids index_right conflict
    )
    
    zcta_merged = zcta_merged.rename(columns={'ZCTA5CE20': 'zip_code'})
    return zcta_merged

In [79]:
def info(df):
    print(" Dataset Info:")
    print(df.info())
    
    print("Shape:", df.shape)
    print("\nMissing Values:")
    print(df.isnull().sum())
    
    print("Data type:")
    print(df['Location'].dtype)
    print(df['Location'].apply(type).value_counts())

df_2022 = pd.read_csv('Real_Estate_Sales_2022.csv')
info(df_2022)


 Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43470 entries, 0 to 43469
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Serial Number     43470 non-null  int64  
 1   List Year         43470 non-null  int64  
 2   Date Recorded     43470 non-null  object 
 3   Town              43470 non-null  object 
 4   Address           43470 non-null  object 
 5   Assessed Value    43470 non-null  float64
 6   Sale Amount       43470 non-null  float64
 7   Sales Ratio       43470 non-null  float64
 8   Property Type     43470 non-null  object 
 9   Residential Type  38965 non-null  object 
 10  Non Use Code      11209 non-null  object 
 11  Assessor Remarks  9756 non-null   object 
 12  OPM remarks       1467 non-null   object 
 13  Location          43468 non-null  object 
dtypes: float64(3), int64(2), object(9)
memory usage: 4.6+ MB
None
Shape: (43470, 14)

Missing Values:
Serial Number        

In [80]:
def clean_addresses(df):
    cleaned = df['Address'].str.lower().str.strip().str.replace('\u00A0', ' ', regex=False)

    # Remove digits, one-letter words, normalize spaces
    cleaned = (
        cleaned
        .str.replace(r'\d+', '', regex=True)
        .str.replace(r'\b[a-zA-Z]\b', '', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
    )

    # Abbreviation replacements
    replacements = {
        r'\bhl\b': 'hill', r'\bhls\b': 'hills', r'\bmdw\b': 'meadow', r'\bmdws\b': 'meadows',
        r'\bvly\b': 'valley', r'\bmtn\b': 'mountain', r'\bmt\b': 'mountain', r'\bmnt\b': 'mount',
        r'\bbrk\b': 'brook', r'\bfld\b': 'field', r'\bbch\b': 'beach', r'\brdg\b': 'ridge',
        r'\bhbr\b': 'harbor', r'\bflds\b': 'fields', r'\bbrg\b': 'bridge', r'\bhlw\b': 'hollow',
        r'\bgrv\b': 'grave', r'\bspg\b': 'spring', r'\bhts\b': 'heights', r'\bml\b': 'mill',
        r'\bpt\b': 'point', r'\bpln\b': 'plain', r'\bplns\b': 'plains', r'\blk\b': 'lake',
        r'\bri\b': 'river', r'\briv\b': 'river', r'\bholw\b': 'hollow', r'\bgr\b': 'grove',
        r'\bfrst\b': 'first', r'\bis\b': 'island', r'\bctr\b': 'center', r'\bvw\b': 'view',
        r'\btrl\b': 'trail', r'\bext\b': 'extension', r'\bcv\b': 'cove', r'\bnck\b': 'neck',
        r'\bhse\b': 'house', r'\bcor\b': 'corner', r'\bbdg\b': 'building', r'\bbnd\b': 'bend',
        r'\bfls\b': 'flats', r'\bknl\b': 'knoll', r'\bcir\b': 'circle', r'\bfrm\b': 'farm',
        r'\bsta\b': 'station', r'\borch\b': 'orchard', r'\bshr\b': 'shrine', r'\bcrk\b': 'creek',
        r'\bln\b': 'lane', r'\bvis\b': 'vista', r'\bbrgs\b': 'boroughs', r'\bspgs\b': 'springs',
        r'\blndg\b': 'landing', r'\bgrn\b': 'green', r'\bun\b': 'union', r'\bmnr\b': 'manor',
        r'\bter\b': 'terrace', r'\bvlg\b': 'village', r'\bxing\b': 'crossing', r'\bfrg\b': 'fringe',
        r'\bdm\b': 'dam', r'\bblfb\b': 'bluff', r'\blts\b': 'lots', r'\brt\b': 'route',
        r'\bgln\b': 'glen', r'\bupr\b': 'upper', r'\bffld\b': 'field',
    }

    for abbr, full in replacements.items():
        cleaned = cleaned.str.replace(abbr, full, regex=True)

    # Strip symbols
    cleaned = (
        cleaned
        .str.replace('multi #', '', regex=False)
        .str.replace('multi#', '', regex=False)
        .str.replace(r'[\/&.\-"\']', '', regex=True)
        .str.replace('  ', ' ', regex=False)
    )

    # Extract only up to st, rd, or ave
    cleaned = cleaned.str.extract(r'^(.*?\b(?:st|rd|ave)\b)', flags=re.IGNORECASE)[0].str.strip()

    df['cleaned_address'] = cleaned
    return df


In [81]:
### Converting and merging OpenAddresses data
def merge(df):
    
    df = clean_addresses(df)

    # Only pull relevant gdf columns
    gdf_subset = gdf[['cleaned_street', 'postcode', 'gdf_geometry']]

    # Merge on normalized street names
    merged = pd.merge(
        df,
        gdf_subset,
        left_on='cleaned_address',
        right_on='cleaned_street',
        how='left'
    )

    # Combine zip_code (df) and postcode (gdf)
    merged['zip_code'] = merged['zip_code'].combine_first(merged['postcode'])

    # Combine location (df) and geometry (gdf)
    merged['Location'] = merged['Location'].combine_first(merged['gdf_geometry'])

    # Drop temporary columns
    merged = merged.drop(columns=['cleaned_street', 'postcode', 'gdf_geometry'])

    return merged



In [82]:
### Loading and Merging all datasets
dataframes = {}
for year in range (2001,2023):
    df = pd.read_csv(f'Real_Estate_Sales_{year}.csv')
    print(f'{year}: ')
    df['geometry'] = df['Location'].apply(parse_location)

    df = convert(df)
    df= merge(df)
    dataframes[year] = df
    

2001: 


C:\Users\tmfmo\AppData\Local\Temp\ipykernel_9084\1440177345.py:5: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: EPSG:4326
Right CRS: EPSG:4269

  zcta_merged = gpd.sjoin(


2002: 


C:\Users\tmfmo\AppData\Local\Temp\ipykernel_9084\1440177345.py:5: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: EPSG:4326
Right CRS: EPSG:4269

  zcta_merged = gpd.sjoin(


2003: 


...

C:\Users\tmfmo\AppData\Local\Temp\ipykernel_9084\1440177345.py:5: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: EPSG:4326
Right CRS: EPSG:4269

  zcta_merged = gpd.sjoin(


2021: 


C:\Users\tmfmo\AppData\Local\Temp\ipykernel_9084\1440177345.py:5: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: EPSG:4326
Right CRS: EPSG:4269

  zcta_merged = gpd.sjoin(


2022: 


C:\Users\tmfmo\AppData\Local\Temp\ipykernel_9084\1440177345.py:5: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: EPSG:4326
Right CRS: EPSG:4269

  zcta_merged = gpd.sjoin(


In [83]:
### Combining datasets to search for potential missing data

# Concatenate all  DataFrames into one DataFrame
all_data = pd.concat(dataframes.values(), ignore_index=True)

# Split missing and non_missing locations
missing_zip_df = all_data[all_data['zip_code'].isnull()].copy()
non_missing_zip_df = all_data[all_data['zip_code'].notnull()].copy()


print(f"Missing zip code rows: {len(missing_zip_df)}")
print(f"Non-missing zip code rows: {len(non_missing_zip_df)}")


Missing zip code rows: 18235
Non-missing zip code rows: 1079394


In [84]:

### Combining dataframes to search for missing data held within other dataframes

address_to_zip = dict(
    zip(non_missing_zip_df['Address'], non_missing_zip_df['zip_code'])
)

# Fill location with previous data
def fill_location(row):
    if pd.isnull(row['zip_code']):
        return address_to_zip.get(row['Address'], None)
    return row['zip_code']

missing_zip_df['Filled_Location'] = missing_zip_df.apply(fill_location, axis=1)

# Update original Location column 
missing_zip_df['zip_code'] = missing_zip_df['Filled_Location']

# Drop columns
missing_zip_df.drop(columns=['Filled_Location', 'Address'], inplace=True)
non_missing_zip_df.drop(columns=['Address'], inplace=True)

# Combine data back into one Dataframe
final_data = pd.concat([non_missing_zip_df, missing_zip_df], ignore_index=True)

print(f"total data: {final_data.shape[0]} rows")
print(f"missing zipcodes: {final_data['zip_code'].isnull().sum()}")

total data: 1097629 rows
missing zipcodes: 17214


In [85]:
### Check for missing data

print("Example df addresses:")
print(dataframes[2001].loc[dataframes[2001]['zip_code'].isna(), 'cleaned_address'].dropna().unique()[:1000])


print("\nExample gdf street names:")
print(gdf['cleaned_street'].dropna().unique()[:1000])


Example df addresses:
['sanford bridge rd' 'chatham field rd' 'suffolk rd' 'river line rd'
 'under mountain rd' 'furdace st' 'burrman rd' 'pope island rd'
 'pavlik rd' 'atta crossing rd' 'woodlawn extension ave' 'palisades ave'
 'cranberry bog rd' 'mansfield grave rd' 'eastbrook heights rd'
 'hillpond rd' 'cucumber hill rd' 'crowpark rd' 'sandy point rd'
 'cherryswamp rd' 'edgemoon rd' 'hiview rd' 'golden rod ave' 'lennox st'
 'borrman rd' 'bertha michael st' 'mountain pleasant st'
 'cedar springs rd' 'green wood st' 'jamaica rd' 'talcott first rd'
 'granby rd' 'haverford rd' 'old tpke rd' 'balm forth st' 'silver sand rd'
 'wheelers farm ave' 'council ring rd' 'talcott view rd' 'ct ave'
 'campville hill rd' 'chatsworth rd' 'spruce mount rd' 'point woods rd'
 'millennium rd' 'fox wood rd' 'aldine st' 'starweather st' 'ridgeway rd'
 'iranstan ave' 'west todd st' 'millpond rd' 'beebe camp rd'
 'old mount rd' 'waterfield st' 'spruce trail hubbell rd' 'east main st'
 'blue hill ave' 'resevo